In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import time
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report

In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [27]:
df = pd.read_parquet('../data/raw/market/df_final.parquet')

In [30]:
df

,begin,ticker,open,close,high,low,value,volume,end
0,2024-01-03 09:00:00,ABIO,103.24,103.24,103.24,103.24,100142.8,970.0,2024-01-03 09:59:59
1,2024-01-03 10:00:00,ABIO,103.00,102.80,103.98,101.62,5349321.4,52030.0,2024-01-03 10:59:59
2,2024-01-03 11:00:00,ABIO,102.84,104.28,104.28,102.54,4188793.6,40410.0,2024-01-03 11:59:59
3,2024-01-03 12:00:00,ABIO,104.22,103.34,104.48,102.88,3192060.2,30840.0,2024-01-03 12:59:59
4,2024-01-03 13:00:00,ABIO,103.40,103.50,103.86,103.06,2405469.8,23230.0,2024-01-03 13:59:59
...,...,...,...,...,...,...,...,...,...
77256,2025-12-09 12:00:00,YDEX,4321.00,4325.50,4329.00,4314.50,98582315.5,22808.0,2025-12-09 12:59:59
77257,2025-12-09 13:00:00,YDEX,4325.50,4316.50,4334.50,4315.50,120564264.5,27876.0,2025-12-09 13:59:59
77258,2025-12-09 14:00:00,YDEX,4316.50,4330.00,4340.50,4315.00,281067170.5,64909.0,2025-12-09 14:59:59
77259,2025-12-09 15:00:00,YDEX,4331.00,4325.00,4334.00,4318.50,147005619.5,33977.0,2025-12-09 15:59:59


In [31]:
df["ticker"].unique()

<ArrowStringArray>
['ABIO', 'AFKS', 'AFLT', 'AKRN', 'ALRS', 'AQUA', 'CHMF', 'GAZP', 'GMKN',
 'HNFG', 'HYDR', 'IRAO', 'LENT', 'MAGN', 'MGNT', 'MRKC', 'MRKV', 'MTSS',
 'MVID', 'NVTK', 'OKEY', 'OZON', 'PHOR', 'PLZL', 'RNFT', 'ROSN', 'RTKM',
 'RUAL', 'SBER', 'SFIN', 'SIBN', 'SNGS', 'SOFL',    'T', 'TATN', 'UGLD',
 'UPRO', 'VKCO', 'VTBR', 'WUSH',   'X5', 'YDEX']
Length: 42, dtype: str

In [33]:
coverage = (
    df.groupby("ticker")["begin"]
    .agg(["min", "max", "count"])
    .sort_values("count")
)

In [34]:
coverage

,min,max,count
ticker,,,
X5,2025-01-09 09:00:00,2025-12-09 16:00:00,4226
OKEY,2024-01-03 09:00:00,2025-12-09 15:00:00,4594
AKRN,2024-01-03 09:00:00,2025-12-09 16:00:00,5590
MRKV,2024-01-03 09:00:00,2025-12-09 16:00:00,5914
MRKC,2024-01-03 09:00:00,2025-12-09 16:00:00,5992
YDEX,2024-07-24 09:00:00,2025-12-09 16:00:00,6004
HNFG,2024-01-03 09:00:00,2025-12-09 16:00:00,6123
LENT,2024-01-03 09:00:00,2025-12-09 16:00:00,6148
OZON,2024-01-03 09:00:00,2025-12-09 16:00:00,6635


In [35]:
SELECTED_TICKERS = [
    "SBER",
    "GAZP",
    "ROSN",
    "NVTK",
    "GMKN",
    "PLZL",
    "TATN",
    "AFLT",
    "YDEX",
    "VKCO",
]

In [36]:
df_selected = df[df["ticker"].isin(SELECTED_TICKERS)].copy()

In [38]:
df_selected = df_selected.sort_values(["ticker", "begin"]).reset_index(drop=True)

In [39]:
df_selected

,begin,ticker,open,close,high,low,value,volume,end
0,2024-01-03 09:00:00,AFLT,35.22,35.22,35.22,35.22,461734.2,13110.0,2024-01-03 09:59:59
1,2024-01-03 10:00:00,AFLT,35.21,35.66,35.67,35.12,41835092.0,1180530.0,2024-01-03 10:59:59
2,2024-01-03 11:00:00,AFLT,35.67,35.48,35.68,35.40,30396905.1,855550.0,2024-01-03 11:59:59
3,2024-01-03 12:00:00,AFLT,35.48,35.55,35.55,35.36,10632512.4,299980.0,2024-01-03 12:59:59
4,2024-01-03 13:00:00,AFLT,35.53,35.59,35.62,35.44,14603209.0,410780.0,2024-01-03 13:59:59
...,...,...,...,...,...,...,...,...,...
79252,2025-12-09 12:00:00,YDEX,4321.00,4325.50,4329.00,4314.50,98582315.5,22808.0,2025-12-09 12:59:59
79253,2025-12-09 13:00:00,YDEX,4325.50,4316.50,4334.50,4315.50,120564264.5,27876.0,2025-12-09 13:59:59
79254,2025-12-09 14:00:00,YDEX,4316.50,4330.00,4340.50,4315.00,281067170.5,64909.0,2025-12-09 14:59:59
79255,2025-12-09 15:00:00,YDEX,4331.00,4325.00,4334.00,4318.50,147005619.5,33977.0,2025-12-09 15:59:59


In [40]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

RAW_MARKET_DIR = PROJECT_ROOT / "data" / "raw" / "market"
PROCESSED_NEWS_DIR = PROJECT_ROOT / "data" / "processed" / "news"

PROCESSED_NEWS_DIR.mkdir(parents=True, exist_ok=True)

SELECTED_TICKERS = [
    "SBER",
    "GAZP",
    "ROSN",
    "NVTK",
    "GMKN",
    "PLZL",
    "TATN",
    "AFLT",
    "YDEX",
    "VKCO",
]

list(RAW_MARKET_DIR.iterdir())

[WindowsPath('C:/Users/HYPERPC/Desktop/ticker-news-feature-pipeline-LLM/data/raw/market/.gitkeep'),
 WindowsPath('C:/Users/HYPERPC/Desktop/ticker-news-feature-pipeline-LLM/data/raw/market/.ipynb_checkpoints'),
 WindowsPath('C:/Users/HYPERPC/Desktop/ticker-news-feature-pipeline-LLM/data/raw/market/df_final.parquet')]

In [41]:
df = pd.read_parquet(RAW_MARKET_DIR / "df_final.parquet")

In [42]:
required_cols = ["begin", "ticker", "open", "close", "high", "low", "value", "volume", "end"]

missing_cols = [col for col in required_cols if col not in df.columns]

missing_cols

[]

In [43]:
df = df.copy()

df["begin"] = pd.to_datetime(df["begin"])
df["end"] = pd.to_datetime(df["end"])

df_selected = df[df["ticker"].isin(SELECTED_TICKERS)].copy()

df_selected = df_selected.sort_values(["ticker", "begin"]).reset_index(drop=True)

df_selected.shape

(79257, 9)

In [44]:
df_selected["ticker"].unique()

<ArrowStringArray>
['AFLT', 'GAZP', 'GMKN', 'NVTK', 'PLZL', 'ROSN', 'SBER', 'TATN', 'VKCO',
 'YDEX']
Length: 10, dtype: str

In [45]:
ticker_universe = (
    df_selected
    .groupby("ticker")
    .agg(
        start_datetime=("begin", "min"),
        end_datetime=("begin", "max"),
        n_rows=("begin", "count"),
    )
    .reset_index()
    .sort_values("ticker")
)

ticker_universe

,ticker,start_datetime,end_datetime,n_rows
0,AFLT,2024-01-03 09:00:00,2025-12-09 16:00:00,8162
1,GAZP,2024-01-03 09:00:00,2025-12-09 16:00:00,8162
2,GMKN,2024-01-03 09:00:00,2025-12-09 16:00:00,8100
3,NVTK,2024-01-03 09:00:00,2025-12-09 16:00:00,8119
4,PLZL,2024-01-03 09:00:00,2025-12-09 16:00:00,8085
5,ROSN,2024-01-03 09:00:00,2025-12-09 16:00:00,8162
6,SBER,2024-01-03 09:00:00,2025-12-09 16:00:00,8162
7,TATN,2024-01-03 09:00:00,2025-12-09 16:00:00,8139
8,VKCO,2024-01-03 09:00:00,2025-12-09 16:00:00,8162
9,YDEX,2024-07-24 09:00:00,2025-12-09 16:00:00,6004


In [46]:
duplicates_count = df_selected.duplicated(["ticker", "begin"]).sum()

duplicates_count

np.int64(0)

In [47]:
market_time_grid = (
    df_selected[["ticker", "begin"]]
    .rename(columns={"begin": "datetime"})
    .drop_duplicates(["ticker", "datetime"])
    .sort_values(["ticker", "datetime"])
    .reset_index(drop=True)
)

market_time_grid.head()

,ticker,datetime
0,AFLT,2024-01-03 09:00:00
1,AFLT,2024-01-03 10:00:00
2,AFLT,2024-01-03 11:00:00
3,AFLT,2024-01-03 12:00:00
4,AFLT,2024-01-03 13:00:00


In [48]:
market_time_grid.shape

(79257, 2)

In [49]:
market_time_grid.groupby("ticker")["datetime"].agg(["min", "max", "count"])

,min,max,count
ticker,,,
AFLT,2024-01-03 09:00:00,2025-12-09 16:00:00,8162
GAZP,2024-01-03 09:00:00,2025-12-09 16:00:00,8162
GMKN,2024-01-03 09:00:00,2025-12-09 16:00:00,8100
NVTK,2024-01-03 09:00:00,2025-12-09 16:00:00,8119
PLZL,2024-01-03 09:00:00,2025-12-09 16:00:00,8085
ROSN,2024-01-03 09:00:00,2025-12-09 16:00:00,8162
SBER,2024-01-03 09:00:00,2025-12-09 16:00:00,8162
TATN,2024-01-03 09:00:00,2025-12-09 16:00:00,8139
VKCO,2024-01-03 09:00:00,2025-12-09 16:00:00,8162


In [50]:
ticker_universe.to_parquet(
    PROCESSED_NEWS_DIR / "ticker_universe.parquet",
    index=False,
)

market_time_grid.to_parquet(
    PROCESSED_NEWS_DIR / "market_time_grid.parquet",
    index=False,
)